# Advanced Data Types & Structures

Python's standard library ships a rich collection of specialised data structures beyond the basic list, dict, and set. Knowing which structure to reach for — and when — is the difference between O(n) and O(1), between readable code and convoluted workarounds.

## What You'll Learn
1. `collections` module deep dive — `Counter`, `defaultdict`, `OrderedDict`, `namedtuple`, `deque`, `ChainMap`, `UserDict`/`UserList`
2. `heapq` — priority queues and heap operations
3. `bisect` — maintaining sorted lists efficiently
4. `array` — typed, compact numeric arrays
5. `struct` — packing/unpacking binary data
6. `enum` — symbolic constants
7. `typing` extras — `TypedDict`, `NamedTuple`, `Literal`, `TypeVar`, `Generic`
8. Trees — building binary search trees
9. Graphs — adjacency lists, BFS, DFS
10. Stacks & queues from scratch
11. Tries (prefix trees)
12. Bloom filters
13. `weakref` — weak references
14. `copy` — shallow vs deep copies
15. Performance comparison — choosing the right structure

---

## 1. `collections` Module — Deep Dive

In [1]:
# ── Counter ───────────────────────────────────────────────────────────────
# A dict subclass for counting hashable objects.
# Missing keys return 0, not KeyError.

from collections import Counter

# Create from iterable
words = "the quick brown fox jumps over the lazy dog the fox".split()
c = Counter(words)
print("Word counts:", c)
print("Most common 3:", c.most_common(3))
print("Missing key (0):", c['cat'])   # 0, not KeyError

# Arithmetic on counters
c2 = Counter({'the': 5, 'fox': 1, 'cat': 3})
print("\nc + c2:", c + c2)       # sum counts
print("c - c2:", c - c2)       # subtract (drop ≤ 0)
print("c & c2:", c & c2)       # intersection (min)
print("c | c2:", c | c2)       # union (max)

# Update and subtract
c.update(['the', 'the', 'cat'])
print("After update:", c['the'], c['cat'])

c.subtract(['the', 'fox'])     # can go negative unlike -
print("After subtract:", c['the'])

# elements() — expand back to a list
small = Counter(a=3, b=1)
print("elements():", sorted(small.elements()))

Word counts: Counter({'the': 3, 'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
Most common 3: [('the', 3), ('fox', 2), ('quick', 1)]
Missing key (0): 0

c + c2: Counter({'the': 8, 'fox': 3, 'cat': 3, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
c - c2: Counter({'quick': 1, 'brown': 1, 'fox': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
c & c2: Counter({'the': 3, 'fox': 1})
c | c2: Counter({'the': 5, 'cat': 3, 'fox': 2, 'quick': 1, 'brown': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1})
After update: 5 1
After subtract: 4
elements(): ['a', 'a', 'a', 'b']


In [2]:
# Counter — practical use cases
import re

text = """
Python is great. Python is versatile. Python is readable.
Great code is readable code. Code in Python.
"""

# Word frequency analysis
words = re.findall(r"\b[a-z]+\b", text.lower())
freq  = Counter(words)

print("Top 5 words:")
for word, count in freq.most_common(5):
    bar = '█' * count
    print(f"  {word:<12} {bar} ({count})")

# Character frequency
letter_freq = Counter(c for c in text.lower() if c.isalpha())
print("\nTop 5 letters:", letter_freq.most_common(5))

# Anagram check using Counter
def is_anagram(s1: str, s2: str) -> bool:
    return Counter(s1.lower()) == Counter(s2.lower())

print(f"\n'listen' & 'silent': {is_anagram('listen', 'silent')}")
print(f"'hello' & 'world':   {is_anagram('hello', 'world')}")

Top 5 words:
  python       ████ (4)
  is           ████ (4)
  code         ███ (3)
  great        ██ (2)
  readable     ██ (2)

Top 5 letters: [('e', 11), ('t', 7), ('o', 7), ('a', 7), ('i', 6)]

'listen' & 'silent': True
'hello' & 'world':   False


In [3]:
# ── defaultdict ───────────────────────────────────────────────────────────
# A dict that calls a factory function to create missing values.
# Eliminates the need for .setdefault() or try/except on missing keys.

from collections import defaultdict

# Group items by a key — classic pattern
employees = [
    ('Alice',  'Engineering'),
    ('Bob',    'Marketing'),
    ('Carol',  'Engineering'),
    ('Dave',   'HR'),
    ('Eve',    'Engineering'),
    ('Frank',  'Marketing'),
]

# Without defaultdict — awkward
by_dept_plain = {}
for name, dept in employees:
    if dept not in by_dept_plain:
        by_dept_plain[dept] = []
    by_dept_plain[dept].append(name)

# With defaultdict — elegant
by_dept = defaultdict(list)
for name, dept in employees:
    by_dept[dept].append(name)   # no KeyError, list created automatically

for dept, names in sorted(by_dept.items()):
    print(f"  {dept:<15}: {', '.join(names)}")

  Engineering    : Alice, Carol, Eve
  HR             : Dave
  Marketing      : Bob, Frank


In [4]:
# defaultdict with different factory types

# int — counting (like Counter but more manual)
freq = defaultdict(int)
for ch in "mississippi":
    freq[ch] += 1
print("Char freq:", dict(sorted(freq.items())))

# set — unique values per key
tag_index = defaultdict(set)
posts = [
    ('Post A', ['python', 'tutorial']),
    ('Post B', ['python', 'advanced']),
    ('Post C', ['tutorial', 'beginner']),
]
for title, tags in posts:
    for tag in tags:
        tag_index[tag].add(title)
print("\nTag index:", dict(tag_index))

# Nested defaultdict — tree-like structures
nested = defaultdict(lambda: defaultdict(int))
nested['row1']['col1'] += 1
nested['row1']['col2'] += 5
nested['row2']['col1'] += 3
print("\nNested defaultdict:")
for row, cols in nested.items():
    print(f"  {row}: {dict(cols)}")

Char freq: {'i': 4, 'm': 1, 'p': 2, 's': 4}

Tag index: {'python': {'Post A', 'Post B'}, 'tutorial': {'Post C', 'Post A'}, 'advanced': {'Post B'}, 'beginner': {'Post C'}}

Nested defaultdict:
  row1: {'col1': 1, 'col2': 5}
  row2: {'col1': 3}


In [5]:
# ── OrderedDict ───────────────────────────────────────────────────────────
# Dicts have been ordered by insertion since Python 3.7,
# but OrderedDict has extra methods: move_to_end() and popitem(last=).

from collections import OrderedDict

od = OrderedDict([('a', 1), ('b', 2), ('c', 3), ('d', 4)])
print("Initial:", list(od.keys()))

od.move_to_end('a')        # move to end
print("After move 'a' to end:", list(od.keys()))

od.move_to_end('d', last=False)  # move to beginning
print("After move 'd' to start:", list(od.keys()))

last  = od.popitem(last=True)    # pop from end (like a stack)
first = od.popitem(last=False)   # pop from start (like a queue)
print(f"Popped last: {last}, first: {first}")
print("Remaining:", dict(od))

# LRU cache implementation using OrderedDict
class LRUCache:
    """Least-Recently-Used cache using OrderedDict."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self._cache   = OrderedDict()

    def get(self, key):
        if key not in self._cache:
            return -1
        self._cache.move_to_end(key)    # mark as recently used
        return self._cache[key]

    def put(self, key, value) -> None:
        if key in self._cache:
            self._cache.move_to_end(key)
        self._cache[key] = value
        if len(self._cache) > self.capacity:
            self._cache.popitem(last=False)   # evict LRU item

    def __repr__(self):
        return f"LRUCache({dict(self._cache)})"


lru = LRUCache(3)
for k, v in [('a',1),('b',2),('c',3)]:
    lru.put(k, v)
print("\nLRU:", lru)
lru.get('a')       # 'a' is now most recent
lru.put('d', 4)    # 'b' should be evicted (least recently used)
print("After access 'a' + add 'd':", lru)

Initial: ['a', 'b', 'c', 'd']
After move 'a' to end: ['b', 'c', 'd', 'a']
After move 'd' to start: ['d', 'b', 'c', 'a']
Popped last: ('a', 1), first: ('d', 4)
Remaining: {'b': 2, 'c': 3}

LRU: LRUCache({'a': 1, 'b': 2, 'c': 3})
After access 'a' + add 'd': LRUCache({'c': 3, 'a': 1, 'd': 4})


In [6]:
# ── namedtuple ────────────────────────────────────────────────────────────
# A tuple subclass with named fields — immutable, memory-efficient,
# and faster than a class with __slots__.

from collections import namedtuple

# Basic namedtuple
Point  = namedtuple('Point',  ['x', 'y'])
Colour = namedtuple('Colour', ['r', 'g', 'b', 'a'], defaults=[255])  # 'a' has default

p = Point(3, 4)
print(f"Point: {p}")
print(f"  x={p.x}, y={p.y}")
print(f"  index 0={p[0]}, index 1={p[1]}")  # tuple access still works

c = Colour(255, 128, 0)    # a defaults to 255
print(f"Colour: {c}")
print(f"  r={c.r}, a={c.a}")

# _replace() — immutable 'update'
p2 = p._replace(x=10)
print(f"Replaced: {p2}")

# _asdict() — convert to OrderedDict
print(f"As dict: {p._asdict()}")

# _fields — field names
print(f"Fields: {Point._fields}")

# Unpack like a regular tuple
x, y = p
print(f"Unpacked: x={x}, y={y}")

Point: Point(x=3, y=4)
  x=3, y=4
  index 0=3, index 1=4
Colour: Colour(r=255, g=128, b=0, a=255)
  r=255, a=255
Replaced: Point(x=10, y=4)
As dict: {'x': 3, 'y': 4}
Fields: ('x', 'y')
Unpacked: x=3, y=4


In [7]:
# namedtuple as lightweight record — CSV parsing example
import csv, io

Employee = namedtuple('Employee', ['name', 'dept', 'salary'])

csv_data = """Alice,Engineering,95000
Bob,Marketing,72000
Carol,Engineering,105000"""

employees = [Employee(*row) for row in csv.reader(io.StringIO(csv_data))]

for emp in employees:
    print(f"  {emp.name:<8} {emp.dept:<15} ${int(emp.salary):>8,}")

# Compute stats using named access
salaries = [int(e.salary) for e in employees]
print(f"Average salary: ${sum(salaries)/len(salaries):,.0f}")

  Alice    Engineering     $  95,000
  Bob      Marketing       $  72,000
  Carol    Engineering     $ 105,000
Average salary: $90,667


In [8]:
# ── deque ─────────────────────────────────────────────────────────────────
# Double-ended queue: O(1) append and pop from BOTH ends.
# list.insert(0, x) and list.pop(0) are O(n) — deque is O(1).

from collections import deque

dq = deque([1, 2, 3, 4, 5], maxlen=7)   # maxlen caps size
print(f"Initial: {dq}")

dq.append(6)          # add to right
dq.appendleft(0)      # add to left
print(f"After append/appendleft: {dq}")

dq.extend([7, 8])     # extend from right (7 fits, 8 evicts 0)
print(f"After extend([7,8]): {dq}")

right = dq.pop()      # remove from right
left  = dq.popleft()  # remove from left
print(f"Popped right={right}, left={left}: {dq}")

# rotate — shift all elements
dq2 = deque([1, 2, 3, 4, 5])
dq2.rotate(2)   # rotate right by 2
print(f"Rotated right 2: {dq2}")
dq2.rotate(-3)  # rotate left by 3
print(f"Rotated left 3:  {dq2}")

# index, count, remove
dq3 = deque([1, 2, 3, 2, 1])
print(f"count(2): {dq3.count(2)}")
dq3.remove(2)   # remove first occurrence
print(f"After remove(2): {dq3}")

Initial: deque([1, 2, 3, 4, 5], maxlen=7)
After append/appendleft: deque([0, 1, 2, 3, 4, 5, 6], maxlen=7)
After extend([7,8]): deque([2, 3, 4, 5, 6, 7, 8], maxlen=7)
Popped right=8, left=2: deque([3, 4, 5, 6, 7], maxlen=7)
Rotated right 2: deque([4, 5, 1, 2, 3])
Rotated left 3:  deque([2, 3, 4, 5, 1])
count(2): 2
After remove(2): deque([1, 3, 2, 1])


In [9]:
# deque use case: sliding window

def sliding_window_max(nums: list[int], k: int) -> list[int]:
    """
    Find the maximum in each sliding window of size k.
    O(n) using a monotonic deque.
    """
    window = deque()   # stores indices, front is always the max
    result = []

    for i, num in enumerate(nums):
        # Remove indices outside the window
        while window and window[0] <= i - k:
            window.popleft()
        # Remove smaller elements (they can never be the max)
        while window and nums[window[-1]] < num:
            window.pop()
        window.append(i)
        if i >= k - 1:
            result.append(nums[window[0]])

    return result


nums = [1, 3, -1, -3, 5, 3, 6, 7]
print(f"Input:           {nums}")
print(f"Sliding max k=3: {sliding_window_max(nums, 3)}")
# Windows: [1,3,-1]=3, [3,-1,-3]=3, [-1,-3,5]=5, [-3,5,3]=5, [5,3,6]=6, [3,6,7]=7

Input:           [1, 3, -1, -3, 5, 3, 6, 7]
Sliding max k=3: [3, 3, 5, 5, 6, 7]


In [10]:
# ── ChainMap ──────────────────────────────────────────────────────────────
# Views multiple dicts as a single mapping.
# Lookups search maps left to right; writes go to the first map.

from collections import ChainMap

# Layered configuration system
defaults      = {'color': 'blue',  'font': 'Arial', 'size': 12, 'bold': False}
theme         = {'color': 'green', 'font': 'Roboto'}
user_settings = {'color': 'red',   'size': 16}

# User settings override theme, which overrides defaults
config = ChainMap(user_settings, theme, defaults)

print("Resolved config:")
for key in sorted(config.keys()):
    source = next(m for m in config.maps if key in m)
    layer  = ['user', 'theme', 'defaults'][config.maps.index(source)]
    print(f"  {key:<8} = {config[key]!r:<10}  (from {layer})")

# Writing goes to the first map (user_settings)
config['bold'] = True
print(f"\nuser_settings after write: {user_settings}")

# new_child — add a higher-priority layer
session_config = config.new_child({'size': 20, 'italic': True})
print(f"Session size: {session_config['size']}")
print(f"Session italic: {session_config['italic']}")

# parents — view without the first map
print(f"Parent size: {config.parents['size']}")

Resolved config:
  bold     = False       (from defaults)
  color    = 'red'       (from user)
  font     = 'Roboto'    (from theme)
  size     = 16          (from user)

user_settings after write: {'color': 'red', 'size': 16, 'bold': True}
Session size: 20
Session italic: True
Parent size: 12


In [11]:
# ── UserDict and UserList ─────────────────────────────────────────────────
# Thin wrappers around dict/list that are designed to be subclassed.
# Safer than subclassing dict/list directly (no edge cases with internal calls).

from collections import UserDict, UserList

class TypedDict(UserDict):
    """A dict that enforces all values must be the same type."""

    def __init__(self, value_type: type, *args, **kwargs):
        self._value_type = value_type
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        if not isinstance(value, self._value_type):
            raise TypeError(
                f"Expected {self._value_type.__name__}, got {type(value).__name__}"
            )
        super().__setitem__(key, value)


scores: TypedDict = TypedDict(int, {'Alice': 90, 'Bob': 85})
scores['Carol'] = 95
print("TypedDict:", dict(scores))

try:
    scores['Dave'] = "excellent"   # should fail
except TypeError as e:
    print(f"TypeError: {e}")


class SortedList(UserList):
    """A list that always stays sorted."""

    def append(self, item):
        super().append(item)
        self.data.sort()

    def insert(self, index, item):
        super().append(item)   # ignore index — maintain sort order
        self.data.sort()


sl = SortedList([3, 1, 4])
sl.append(2)
sl.insert(0, 5)
print("SortedList:", sl)

TypedDict: {'Alice': 90, 'Bob': 85, 'Carol': 95}
TypeError: Expected int, got str
SortedList: [1, 2, 3, 4, 5]


---

## 2. `heapq` — Priority Queues

In [12]:
import heapq

# Python's heapq implements a min-heap.
# The smallest element is always at index 0.

heap = []
for val in [5, 1, 8, 3, 9, 2, 7, 4, 6]:
    heapq.heappush(heap, val)

print(f"Heap array: {heap}")
print(f"Peek min:   {heap[0]}")

# Pop elements in sorted order
sorted_vals = [heapq.heappop(heap) for _ in range(len(heap))]
print(f"Popped in order: {sorted_vals}")

# heapify — convert an existing list to a heap in O(n)
data = [5, 1, 8, 3, 9, 2, 7]
heapq.heapify(data)
print(f"Heapified: {data}")

# nlargest / nsmallest — O(n log k) — faster than sorting for small k
data = [5, 1, 8, 3, 9, 2, 7, 4, 6]
print(f"3 largest:  {heapq.nlargest(3, data)}")
print(f"3 smallest: {heapq.nsmallest(3, data)}")

# pushpop and heapreplace — atomic operations (more efficient than push+pop)
heap = [1, 3, 5, 7, 9]
heapq.heapify(heap)
result = heapq.heappushpop(heap, 4)   # push 4, then pop min
print(f"pushpop(4): returned {result}, heap={heap}")

Heap array: [1, 3, 2, 4, 9, 8, 7, 5, 6]
Peek min:   1
Popped in order: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Heapified: [1, 3, 2, 5, 9, 8, 7]
3 largest:  [9, 8, 7]
3 smallest: [1, 2, 3]
pushpop(4): returned 1, heap=[3, 4, 5, 7, 9]


In [13]:
# Max-heap — negate values to simulate
max_heap = []
for val in [5, 1, 8, 3, 9, 2, 7]:
    heapq.heappush(max_heap, -val)   # store negated

print("Max-heap pops:", end=" ")
while max_heap:
    print(-heapq.heappop(max_heap), end=" ")
print()

# Priority queue with (priority, item) tuples
task_queue = []
heapq.heappush(task_queue, (3, 'low priority task'))
heapq.heappush(task_queue, (1, 'critical task'))
heapq.heappush(task_queue, (2, 'medium priority task'))
heapq.heappush(task_queue, (1, 'another critical task'))

print("\nProcessing tasks by priority:")
while task_queue:
    priority, task = heapq.heappop(task_queue)
    print(f"  [{priority}] {task}")

Max-heap pops: 9 8 7 5 3 2 1 

Processing tasks by priority:
  [1] another critical task
  [1] critical task
  [2] medium priority task
  [3] low priority task


In [14]:
# Full PriorityQueue class wrapping heapq
import itertools

class PriorityQueue:
    """
    Min-priority queue using heapq.
    Handles equal priorities correctly using a tie-breaker counter.
    """

    def __init__(self):
        self._heap    = []
        self._counter = itertools.count()   # unique tie-breaker

    def push(self, item, priority: int = 0) -> None:
        count = next(self._counter)
        heapq.heappush(self._heap, (priority, count, item))

    def pop(self):
        if not self._heap:
            raise IndexError("pop from empty PriorityQueue")
        _, _, item = heapq.heappop(self._heap)
        return item

    def peek(self):
        return self._heap[0][2] if self._heap else None

    def __len__(self) -> int:
        return len(self._heap)

    def __bool__(self) -> bool:
        return bool(self._heap)


pq = PriorityQueue()
pq.push('email',   priority=3)
pq.push('bug fix', priority=1)
pq.push('meeting', priority=2)
pq.push('hotfix',  priority=1)   # same priority as 'bug fix' — FIFO order

print("Tasks by priority:")
while pq:
    print(f"  {pq.pop()}")

Tasks by priority:
  bug fix
  hotfix
  meeting
  email


---

## 3. `bisect` — Maintaining Sorted Lists

In [15]:
import bisect

# bisect_left / bisect_right — find insertion point in sorted list
# bisect_left:  insert BEFORE existing equal elements
# bisect_right: insert AFTER  existing equal elements (default)

data = [1, 3, 5, 5, 5, 7, 9, 11]
print(f"data: {data}")
print(f"bisect_left(5):  {bisect.bisect_left(data, 5)}")   # → 2
print(f"bisect_right(5): {bisect.bisect_right(data, 5)}")  # → 5
print(f"bisect_left(6):  {bisect.bisect_left(data, 6)}")   # → 5 (between 5s and 7)

# insort — insert while keeping sorted order, O(n) due to list shift
sorted_nums = [1, 3, 5, 7, 9]
bisect.insort(sorted_nums, 4)
bisect.insort(sorted_nums, 6)
print(f"\nAfter insort(4) and insort(6): {sorted_nums}")

# Efficient membership in sorted list
def sorted_contains(sorted_list: list, value) -> bool:
    idx = bisect.bisect_left(sorted_list, value)
    return idx < len(sorted_list) and sorted_list[idx] == value

print(f"\nContains 5: {sorted_contains(data, 5)}")
print(f"Contains 6: {sorted_contains(data, 6)}")

data: [1, 3, 5, 5, 5, 7, 9, 11]
bisect_left(5):  2
bisect_right(5): 5
bisect_left(6):  5

After insort(4) and insort(6): [1, 3, 4, 5, 6, 7, 9]

Contains 5: True
Contains 6: False


In [16]:
# Grade lookup with bisect — a classic use case
breakpoints = [60, 70, 80, 90]
grades      = ['F', 'D', 'C', 'B', 'A']

def get_grade(score: int) -> str:
    return grades[bisect.bisect_right(breakpoints, score)]

test_scores = [45, 62, 75, 82, 91, 100]
for score in test_scores:
    print(f"  {score:3d} → {get_grade(score)}")

# Range queries on sorted data
data = sorted([5, 3, 8, 1, 9, 2, 7, 4, 6, 3, 5])

def count_in_range(sorted_list: list, lo, hi) -> int:
    """Count elements in [lo, hi] using two binary searches."""
    left  = bisect.bisect_left(sorted_list, lo)
    right = bisect.bisect_right(sorted_list, hi)
    return right - left

print(f"\ndata: {data}")
print(f"count in [3, 6]: {count_in_range(data, 3, 6)}")

   45 → F
   62 → D
   75 → C
   82 → B
   91 → A
  100 → A

data: [1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
count in [3, 6]: 6


---

## 4. `array` — Typed Numeric Arrays

In [17]:
import array, sys

# array.array stores values as typed C arrays — much less memory than a list.
# Type codes: 'b'=int8, 'B'=uint8, 'h'=int16, 'i'=int32, 'l'=int64,
#             'f'=float32, 'd'=float64

py_list  = list(range(1_000_000))
int_arr  = array.array('i', range(1_000_000))   # 32-bit ints
float_arr= array.array('f', range(1_000_000))   # 32-bit floats

print(f"list     memory: {sys.getsizeof(py_list):>10,} bytes")
print(f"array(i) memory: {int_arr.buffer_info()[1] * int_arr.itemsize:>10,} bytes")
print(f"array(f) memory: {float_arr.buffer_info()[1] * float_arr.itemsize:>10,} bytes")

# Operations
a = array.array('d', [1.0, 2.5, 3.7, 4.1])
print(f"\narray: {a}")
print(f"type code: {a.typecode}")
print(f"item size: {a.itemsize} bytes")
print(f"[1]: {a[1]}")

a.append(5.9)
a.extend([6.1, 7.2])
a.insert(0, 0.5)
print(f"After operations: {a}")

# Efficient file I/O
import tempfile, os
tmp = tempfile.mktemp()
with open(tmp, 'wb') as f:
    a.tofile(f)

b = array.array('d')
with open(tmp, 'rb') as f:
    b.fromfile(f, len(a))
print(f"Loaded from file: {b}")
os.unlink(tmp)

list     memory:  8,000,056 bytes
array(i) memory:  4,000,000 bytes
array(f) memory:  4,000,000 bytes

array: array('d', [1.0, 2.5, 3.7, 4.1])
type code: d
item size: 8 bytes
[1]: 2.5
After operations: array('d', [0.5, 1.0, 2.5, 3.7, 4.1, 5.9, 6.1, 7.2])
Loaded from file: array('d', [0.5, 1.0, 2.5, 3.7, 4.1, 5.9, 6.1, 7.2])


---

## 5. `enum` — Symbolic Constants

In [18]:
from enum import Enum, IntEnum, Flag, IntFlag, auto, unique

# Basic Enum
class Direction(Enum):
    NORTH = 'N'
    SOUTH = 'S'
    EAST  = 'E'
    WEST  = 'W'

d = Direction.NORTH
print(f"name:  {d.name}")
print(f"value: {d.value}")
print(f"repr:  {d!r}")

# Lookup by value
print(f"\nLookup by value: {Direction('S')}")

# Iteration
print("All directions:", list(Direction))

# Comparison — identity, not equality
print(f"d is NORTH: {d is Direction.NORTH}")
print(f"d == 'N':   {d == 'N'}")   # False — Enum members are not their values

name:  NORTH
value: N
repr:  <Direction.NORTH: 'N'>

Lookup by value: Direction.SOUTH
All directions: [<Direction.NORTH: 'N'>, <Direction.SOUTH: 'S'>, <Direction.EAST: 'E'>, <Direction.WEST: 'W'>]
d is NORTH: True
d == 'N':   False


In [19]:
# auto() — auto-assign values
class Color(Enum):
    RED   = auto()   # 1
    GREEN = auto()   # 2
    BLUE  = auto()   # 3

print([c.value for c in Color])

# IntEnum — inherits from int; can be used where ints are expected
class HTTPStatus(IntEnum):
    OK          = 200
    CREATED     = 201
    BAD_REQUEST = 400
    NOT_FOUND   = 404
    SERVER_ERROR= 500

status = HTTPStatus.OK
print(f"\n{status}: {status.name}")
print(f"Is success (200-299): {200 <= status < 300}")

# Flag — combinable bit flags
class Permission(Flag):
    READ    = auto()
    WRITE   = auto()
    EXECUTE = auto()
    ALL     = READ | WRITE | EXECUTE

user_perms = Permission.READ | Permission.WRITE
print(f"\nUser permissions: {user_perms}")
print(f"Can read:    {Permission.READ  in user_perms}")
print(f"Can execute: {Permission.EXECUTE in user_perms}")

# Enum with methods
@unique   # ensures no duplicate values
class Planet(Enum):
    MERCURY = (3.303e+23, 2.4397e6)
    VENUS   = (4.869e+24, 6.0518e6)
    EARTH   = (5.976e+24, 6.37814e6)
    MARS    = (6.421e+23, 3.3972e6)

    def __init__(self, mass, radius):
        self.mass   = mass
        self.radius = radius

    @property
    def surface_gravity(self) -> float:
        G = 6.67430e-11
        return G * self.mass / self.radius ** 2

for planet in Planet:
    print(f"  {planet.name:<8} g={planet.surface_gravity:.2f} m/s²")

[1, 2, 3]

200: OK
Is success (200-299): True

User permissions: Permission.READ|WRITE
Can read:    True
Can execute: False
  MERCURY  g=3.70 m/s²
  VENUS    g=8.87 m/s²
  EARTH    g=9.80 m/s²
  MARS     g=3.71 m/s²


---

## 6. `typing` Extras — `TypedDict`, `NamedTuple`, `TypeVar`, `Generic`

In [20]:
from typing import TypedDict, NamedTuple, Literal, Required, NotRequired

# TypedDict — a dict with type annotations for each key
class Movie(TypedDict):
    title:    str
    year:     int
    rating:   float
    director: NotRequired[str]   # optional key (Python 3.11+)

m: Movie = {'title': 'Inception', 'year': 2010, 'rating': 8.8}
print(f"Movie: {m}")

# Total=False — all keys optional
class Config(TypedDict, total=False):
    host:    str
    port:    int
    debug:   bool

cfg: Config = {'host': 'localhost'}   # only one key — valid
print(f"Config: {cfg}")

# Literal — restrict to specific values
def set_mode(mode: Literal['read', 'write', 'append']) -> None:
    print(f"Mode set to: {mode}")

set_mode('read')
# set_mode('delete')  # mypy would flag this as an error

Movie: {'title': 'Inception', 'year': 2010, 'rating': 8.8}
Config: {'host': 'localhost'}
Mode set to: read


In [21]:
# typing.NamedTuple — class-based namedtuple with type hints and defaults
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat:  float
    lon:  float
    alt:  float = 0.0   # default value

    @property
    def is_northern(self) -> bool:
        return self.lat > 0

nyc = Coordinate(40.7128, -74.0060)
print(f"NYC: {nyc}")
print(f"Northern hemisphere: {nyc.is_northern}")
print(f"Tuple access: lat={nyc[0]}")
print(f"Fields: {Coordinate._fields}")

NYC: Coordinate(lat=40.7128, lon=-74.006, alt=0.0)
Northern hemisphere: True
Tuple access: lat=40.7128
Fields: ('lat', 'lon', 'alt')


In [22]:
# TypeVar and Generic — parameterised / generic classes
from typing import TypeVar, Generic

T = TypeVar('T')
K = TypeVar('K')
V = TypeVar('V')


class Stack(Generic[T]):
    """A type-safe stack."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self) -> T:
        if not self._items:
            raise IndexError("peek at empty stack")
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)

    def __bool__(self) -> bool:
        return bool(self._items)

    def __repr__(self) -> str:
        return f"Stack({self._items!r})"


int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)
print(f"Stack: {int_stack}")
print(f"Peek: {int_stack.peek()}")
print(f"Pop: {int_stack.pop()}")
print(f"After pop: {int_stack}")

Stack: Stack([1, 2, 3])
Peek: 3
Pop: 3
After pop: Stack([1, 2])


---

## 7. Trees — Binary Search Tree

In [23]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional, Iterator


@dataclass
class BSTNode:
    value: int
    left:  Optional[BSTNode] = field(default=None, repr=False)
    right: Optional[BSTNode] = field(default=None, repr=False)


class BinarySearchTree:
    """Binary search tree with insert, search, delete, and traversals."""

    def __init__(self):
        self.root: Optional[BSTNode] = None

    def insert(self, value: int) -> None:
        self.root = self._insert(self.root, value)

    def _insert(self, node: Optional[BSTNode], value: int) -> BSTNode:
        if node is None:
            return BSTNode(value)
        if value < node.value:
            node.left  = self._insert(node.left,  value)
        elif value > node.value:
            node.right = self._insert(node.right, value)
        return node

    def search(self, value: int) -> bool:
        node = self.root
        while node:
            if value == node.value:  return True
            elif value < node.value: node = node.left
            else:                    node = node.right
        return False

    def delete(self, value: int) -> None:
        self.root = self._delete(self.root, value)

    def _delete(self, node: Optional[BSTNode], value: int) -> Optional[BSTNode]:
        if node is None:
            return None
        if value < node.value:
            node.left  = self._delete(node.left,  value)
        elif value > node.value:
            node.right = self._delete(node.right, value)
        else:   # found the node to delete
            if node.left is None:  return node.right
            if node.right is None: return node.left
            # Two children: replace with in-order successor
            successor = node.right
            while successor.left:
                successor = successor.left
            node.value = successor.value
            node.right = self._delete(node.right, successor.value)
        return node

    def inorder(self) -> list[int]:
        """Left → Root → Right: produces sorted output."""
        result = []
        def _inorder(n):
            if n:
                _inorder(n.left)
                result.append(n.value)
                _inorder(n.right)
        _inorder(self.root)
        return result

    def height(self) -> int:
        def _height(n):
            if n is None: return 0
            return 1 + max(_height(n.left), _height(n.right))
        return _height(self.root)

    def __contains__(self, value: int) -> bool:
        return self.search(value)

    def visualise(self, node=None, indent="", last=True):
        """Print a simple ASCII tree."""
        if node is None:
            node = self.root
        if node:
            print(indent + ("└─ " if last else "├─ ") + str(node.value))
            children = [c for c in [node.left, node.right] if c]
            for i, child in enumerate(children):
                is_last = i == len(children) - 1
                new_indent = indent + ("   " if last else "│  ")
                self.visualise(child, new_indent, is_last)


bst = BinarySearchTree()
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    bst.insert(v)

print("BST structure:")
bst.visualise()
print(f"\nInorder (sorted): {bst.inorder()}")
print(f"Height: {bst.height()}")
print(f"6 in BST: {6 in bst}")
print(f"5 in BST: {5 in bst}")

bst.delete(3)
print(f"\nAfter deleting 3: {bst.inorder()}")

BST structure:
└─ 8
   ├─ 3
   │  ├─ 1
   │  └─ 6
   │     ├─ 4
   │     └─ 7
   └─ 10
      └─ 14
         └─ 13

Inorder (sorted): [1, 3, 4, 6, 7, 8, 10, 13, 14]
Height: 4
6 in BST: True
5 in BST: False

After deleting 3: [1, 4, 6, 7, 8, 10, 13, 14]


---

## 8. Graphs — Adjacency Lists, BFS, DFS

In [24]:
from collections import defaultdict, deque
from typing import Optional


class Graph:
    """Undirected weighted graph using adjacency lists."""

    def __init__(self, directed: bool = False):
        self.directed = directed
        self._adj: dict[str, list[tuple[str, float]]] = defaultdict(list)

    def add_edge(self, u: str, v: str, weight: float = 1.0) -> None:
        self._adj[u].append((v, weight))
        if not self.directed:
            self._adj[v].append((u, weight))

    def neighbours(self, node: str) -> list[tuple[str, float]]:
        return self._adj[node]

    @property
    def nodes(self) -> set[str]:
        nodes = set(self._adj.keys())
        for neighbours in self._adj.values():
            nodes.update(n for n, _ in neighbours)
        return nodes

    def bfs(self, start: str) -> list[str]:
        """Breadth-First Search — explores level by level."""
        visited = {start}
        queue   = deque([start])
        order   = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for neighbour, _ in sorted(self._adj[node]):
                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append(neighbour)
        return order

    def dfs(self, start: str) -> list[str]:
        """Depth-First Search — explores as deep as possible first."""
        visited = set()
        order   = []
        def _dfs(node: str) -> None:
            visited.add(node)
            order.append(node)
            for neighbour, _ in sorted(self._adj[node]):
                if neighbour not in visited:
                    _dfs(neighbour)
        _dfs(start)
        return order

    def shortest_path(self, start: str, end: str) -> Optional[list[str]]:
        """Dijkstra's algorithm — shortest weighted path."""
        import heapq
        dist  = {node: float('inf') for node in self.nodes}
        prev  = {}
        dist[start] = 0
        heap  = [(0, start)]

        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]:
                continue
            for v, w in self._adj[u]:
                alt = dist[u] + w
                if alt < dist[v]:
                    dist[v] = alt
                    prev[v] = u
                    heapq.heappush(heap, (alt, v))

        if end not in prev and end != start:
            return None
        path = []
        node = end
        while node != start:
            path.append(node)
            node = prev[node]
        path.append(start)
        return list(reversed(path)), dist[end]


g = Graph()
edges = [
    ('A', 'B', 4), ('A', 'C', 2), ('B', 'C', 1), ('B', 'D', 5),
    ('C', 'D', 8), ('C', 'E', 10), ('D', 'E', 2), ('D', 'F', 6),
    ('E', 'F', 3),
]
for u, v, w in edges:
    g.add_edge(u, v, w)

print("BFS from A:", g.bfs('A'))
print("DFS from A:", g.dfs('A'))
path, dist = g.shortest_path('A', 'F')
print(f"Shortest A→F: {' → '.join(path)} (cost={dist})")

BFS from A: ['A', 'B', 'C', 'D', 'E', 'F']
DFS from A: ['A', 'B', 'C', 'D', 'E', 'F']
Shortest A→F: A → C → B → D → E → F (cost=13)


---

## 9. Tries (Prefix Trees)

In [25]:
class TrieNode:
    __slots__ = ('children', 'is_end', 'count')

    def __init__(self):
        self.children: dict[str, TrieNode] = {}
        self.is_end:   bool = False
        self.count:    int  = 0   # words with this prefix


class Trie:
    """
    Prefix tree for fast string search and autocomplete.
    Insert/search/startswith are all O(m) where m = word length.
    """

    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.count += 1
        node.is_end = True

    def search(self, word: str) -> bool:
        """Return True if word is in the trie."""
        node = self._traverse(word)
        return node is not None and node.is_end

    def starts_with(self, prefix: str) -> bool:
        """Return True if any word starts with prefix."""
        return self._traverse(prefix) is not None

    def count_with_prefix(self, prefix: str) -> int:
        """Return the number of words with the given prefix."""
        node = self._traverse(prefix)
        return node.count if node else 0

    def autocomplete(self, prefix: str) -> list[str]:
        """Return all words starting with prefix."""
        node = self._traverse(prefix)
        if node is None:
            return []
        results = []
        self._collect(node, prefix, results)
        return sorted(results)

    def _traverse(self, prefix: str) -> Optional[TrieNode]:
        node = self.root
        for char in prefix:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def _collect(self, node: TrieNode, prefix: str, results: list) -> None:
        if node.is_end:
            results.append(prefix)
        for char, child in node.children.items():
            self._collect(child, prefix + char, results)


trie = Trie()
words = ['apple', 'app', 'application', 'apply', 'apt', 'banana', 'band', 'bandana']
for word in words:
    trie.insert(word)

print("Search:")
for w in ['app', 'apple', 'ap', 'cat']:
    print(f"  {w!r:<15}: {trie.search(w)}")

print("\nAutocomplete 'app':")
print(f"  {trie.autocomplete('app')}")

print("\nAutocomplete 'ban':")
print(f"  {trie.autocomplete('ban')}")

print(f"\nWords with prefix 'ap': {trie.count_with_prefix('ap')}")

Search:
  'app'          : True
  'apple'        : True
  'ap'           : False
  'cat'          : False

Autocomplete 'app':
  ['app', 'apple', 'application', 'apply']

Autocomplete 'ban':
  ['banana', 'band', 'bandana']

Words with prefix 'ap': 5


---

## 10. Bloom Filters

In [26]:
import hashlib
import math


class BloomFilter:
    """
    Probabilistic set membership test.

    - "definitely not in set" is always correct
    - "possibly in set" has a small false-positive rate
    - Uses far less memory than a set for large collections
    """

    def __init__(self, capacity: int, false_positive_rate: float = 0.01):
        self.capacity    = capacity
        self.fpr         = false_positive_rate
        # Optimal bit array size and number of hash functions
        self.size        = self._optimal_size(capacity, false_positive_rate)
        self.num_hashes  = self._optimal_hashes(self.size, capacity)
        self.bit_array   = bytearray(self.size)
        self._count      = 0

    @staticmethod
    def _optimal_size(n: int, p: float) -> int:
        return int(-(n * math.log(p)) / (math.log(2) ** 2))

    @staticmethod
    def _optimal_hashes(m: int, n: int) -> int:
        return max(1, int((m / n) * math.log(2)))

    def _hash_positions(self, item: str) -> list[int]:
        positions = []
        for i in range(self.num_hashes):
            digest = hashlib.md5(f"{item}:{i}".encode()).hexdigest()
            positions.append(int(digest, 16) % self.size)
        return positions

    def add(self, item: str) -> None:
        for pos in self._hash_positions(item):
            self.bit_array[pos] = 1
        self._count += 1

    def __contains__(self, item: str) -> bool:
        return all(self.bit_array[pos] for pos in self._hash_positions(item))

    @property
    def estimated_fpr(self) -> float:
        """Estimate current false positive rate based on fill level."""
        fill = sum(self.bit_array) / self.size
        return fill ** self.num_hashes


bf = BloomFilter(1000, false_positive_rate=0.01)
print(f"Bit array size: {bf.size:,} bits ({bf.size//8} bytes)")
print(f"Num hash funcs: {bf.num_hashes}")

# Add words
for word in ['python', 'java', 'rust', 'go', 'c++']:
    bf.add(word)

# Test membership
for word in ['python', 'ruby', 'java', 'swift']:
    result = "definitely not in" if word not in bf else "possibly in"
    print(f"  {word:<10}: {result}")

print(f"\nEstimated FPR: {bf.estimated_fpr:.4%}")

# Memory comparison with a set
import sys
big_words = [f"word_{i}" for i in range(10_000)]
big_bf = BloomFilter(10_000)
big_set = set()
for w in big_words:
    big_bf.add(w)
    big_set.add(w)

print(f"\n10,000 words:")
print(f"  BloomFilter: {sys.getsizeof(big_bf.bit_array):>8,} bytes")
print(f"  set:         {sys.getsizeof(big_set):>8,} bytes")

Bit array size: 9,585 bits (1198 bytes)
Num hash funcs: 6
  python    : possibly in
  ruby      : definitely not in
  java      : possibly in
  swift     : definitely not in

Estimated FPR: 0.0000%

10,000 words:
  BloomFilter:   95,907 bytes
  set:          524,504 bytes


---

## 11. `weakref` — Weak References

In [27]:
import weakref
import gc

# A weak reference does NOT prevent garbage collection.
# When the only remaining references are weak, the object is collected.

class Widget:
    def __init__(self, name: str):
        self.name = name

    def __del__(self):
        print(f"  Widget '{self.name}' deleted")


# Strong reference — keeps object alive
w = Widget("button")
ref = weakref.ref(w)      # weak reference

print(f"ref() while alive: {ref()}")   # returns the object
print(f"Name: {ref().name}")

print("\nDeleting strong reference...")
del w
gc.collect()              # force collection

print(f"ref() after delete: {ref()}")  # returns None

# WeakValueDictionary — cache that doesn't prevent GC
class Cache:
    _cache: weakref.WeakValueDictionary = weakref.WeakValueDictionary()

    @classmethod
    def get_or_create(cls, key: str) -> Widget:
        if key not in cls._cache:
            print(f"  Creating Widget('{key}')")
            cls._cache[key] = Widget(key)
        else:
            print(f"  Cache hit for '{key}'")
        return cls._cache[key]


print("\nWeak value cache:")
a = Cache.get_or_create('slider')    # creates
b = Cache.get_or_create('slider')    # cache hit
print(f"Same object: {a is b}")

print("Releasing strong refs...")
del a, b
gc.collect()

c = Cache.get_or_create('slider')    # creates again — was GC'd
del c

ref() while alive: <__main__.Widget object at 0x10cc63230>
Name: button

Deleting strong reference...
  Widget 'button' deleted
ref() after delete: None

Weak value cache:
  Creating Widget('slider')
  Widget 'slider' deleted


KeyError: 'slider'

---

## 12. `copy` — Shallow vs Deep Copies

In [ ]:
import copy

# Assignment — just another name for the same object
original = [[1, 2], [3, 4], [5, 6]]
alias    = original          # same object
shallow  = copy.copy(original)   # new list, same inner lists
deep     = copy.deepcopy(original)  # fully independent

print(f"original is alias:   {original is alias}")
print(f"original is shallow: {original is shallow}")
print(f"original is deep:    {original is deep}")
print(f"original[0] is shallow[0]: {original[0] is shallow[0]}")  # True
print(f"original[0] is deep[0]:    {original[0] is deep[0]}")     # False

# Modifying an inner object
original[0].append(99)
print(f"\nAfter original[0].append(99):")
print(f"  alias[0]:   {alias[0]}"   )  # [1, 2, 99] — same object
print(f"  shallow[0]: {shallow[0]}")   # [1, 2, 99] — shared inner list!
print(f"  deep[0]:    {deep[0]}"   )   # [1, 2]     — independent

# Adding to the outer list
original.append([7, 8])
print(f"\nAfter original.append([7,8]):")
print(f"  alias:   {len(alias)} items")    # 4
print(f"  shallow: {len(shallow)} items")  # 3 — shallow is a different list
print(f"  deep:    {len(deep)} items")     # 3

In [ ]:
# Custom copy behaviour using __copy__ and __deepcopy__

class Config:
    def __init__(self, settings: dict, readonly: bool = False):
        self.settings = settings
        self.readonly = readonly
        self._cache   = {}   # internal cache — should NOT be copied

    def __copy__(self):
        """Shallow copy shares the settings dict."""
        new = Config.__new__(Config)
        new.settings = self.settings   # shared reference
        new.readonly = self.readonly
        new._cache   = {}              # fresh cache (not shared)
        return new

    def __deepcopy__(self, memo):
        """Deep copy creates independent settings."""
        new = Config.__new__(Config)
        memo[id(self)] = new           # register in memo to handle cycles
        new.settings = copy.deepcopy(self.settings, memo)
        new.readonly = self.readonly
        new._cache   = {}              # always fresh
        return new


cfg    = Config({'db': {'host': 'localhost', 'port': 5432}})
shal   = copy.copy(cfg)
deep   = copy.deepcopy(cfg)

cfg.settings['db']['port'] = 9999
print(f"Original port: {cfg.settings['db']['port']}")
print(f"Shallow  port: {shal.settings['db']['port']}")  # 9999 — shared!
print(f"Deep     port: {deep.settings['db']['port']}")  # 5432 — independent

---

## 13. Performance Comparison — Choosing the Right Structure

In [ ]:
import timeit, random

N = 100_000
data = list(range(N))
random.shuffle(data)
target = N // 2

# ── Search comparison ─────────────────────────────────────────────────────
lst   = list(data)
st    = set(data)
dct   = {v: v for v in data}

t_list = timeit.timeit(lambda: target in lst, number=1000)
t_set  = timeit.timeit(lambda: target in st,  number=1000)
t_dict = timeit.timeit(lambda: target in dct, number=1000)

print("Search for element in 100k items (1000 lookups):")
print(f"  list:   {t_list*1000:.2f}ms   O(n)")
print(f"  set:    {t_set*1000:.4f}ms  O(1) average")
print(f"  dict:   {t_dict*1000:.4f}ms  O(1) average")

In [ ]:
from collections import deque

# ── Front insertion comparison ────────────────────────────────────────────
t_list_insert = timeit.timeit(lambda: lst.insert(0, -1), number=1000)
dq = deque(data)
t_deque_left  = timeit.timeit(lambda: dq.appendleft(-1), number=1000)

print("Front insertion (1000 ops):")
print(f"  list.insert(0, x): {t_list_insert*1000:.2f}ms   O(n) — shifts all elements")
print(f"  deque.appendleft:  {t_deque_left*1000:.4f}ms  O(1)")

In [ ]:
# ── Big-O complexity reference ────────────────────────────────────────────
complexity = """
╔══════════════╦══════════╦══════════╦══════════╦═════════════════════════════╗
║ Structure    ║ Access   ║ Search   ║ Insert   ║ Notes                       ║
╠══════════════╬══════════╬══════════╬══════════╬═════════════════════════════╣
║ list         ║ O(1)     ║ O(n)     ║ O(n)*    ║ *O(1) amortised at end      ║
║ deque        ║ O(n)     ║ O(n)     ║ O(1)     ║ O(1) both ends              ║
║ dict         ║ O(1)     ║ O(1)     ║ O(1)     ║ hash table; worst O(n)      ║
║ set          ║ —        ║ O(1)     ║ O(1)     ║ hash table; worst O(n)      ║
║ sorted list  ║ O(1)     ║ O(log n) ║ O(n)     ║ binary search via bisect    ║
║ heapq        ║ O(1) min ║ O(n)     ║ O(log n) ║ min at index 0              ║
║ BST          ║ O(log n) ║ O(log n) ║ O(log n) ║ O(n) worst if unbalanced    ║
║ trie         ║ O(m)     ║ O(m)     ║ O(m)     ║ m = word length             ║
║ Counter      ║ O(1)     ║ O(1)     ║ O(1)     ║ dict subclass               ║
║ OrderedDict  ║ O(1)     ║ O(1)     ║ O(1)     ║ move_to_end O(1)            ║
║ array.array  ║ O(1)     ║ O(n)     ║ O(n)     ║ more memory-efficient       ║
╚══════════════╩══════════╩══════════╩══════════╩═════════════════════════════╝
"""
print(complexity)

In [ ]:
# ── Choosing the right structure ──────────────────────────────────────────
guide = """
CHOOSING A DATA STRUCTURE
==========================

Need fast membership testing?
  → set or dict (O(1) average)

Need ordered insertion + both-end access?
  → deque

Need always-sorted iteration?
  → sorted list + bisect.insort (inserts stay sorted)
  → heapq (only if you just need the minimum/maximum repeatedly)

Need to count things?
  → Counter

Need to group items by key?
  → defaultdict(list)

Need layered config / scoped lookups?
  → ChainMap

Need immutable record with field names?
  → namedtuple or typing.NamedTuple

Need mutable record with auto __init__/__repr__?
  → @dataclass

Need efficient typed numeric arrays?
  → array.array (or numpy for math)

Need prefix search / autocomplete?
  → Trie

Need approximate membership with low memory?
  → BloomFilter

Need priority ordering?
  → heapq or PriorityQueue

Need LRU cache?
  → functools.lru_cache or OrderedDict-based LRUCache

Need cache that auto-evicts when memory is tight?
  → weakref.WeakValueDictionary
"""
print(guide)

In [ ]:
# 🏆 Practice: build a frequency-ranked autocomplete system
# combining Trie + Counter + heapq

class AutoComplete:
    """
    Autocomplete engine that ranks suggestions by frequency.
    Uses a Trie for prefix lookup and a Counter for ranking.
    """

    def __init__(self):
        self.trie    = Trie()
        self.counter = Counter()

    def add(self, word: str, count: int = 1) -> None:
        """Add a word with an optional frequency count."""
        self.trie.insert(word)
        self.counter[word] += count

    def suggest(self, prefix: str, top_k: int = 5) -> list[tuple[str, int]]:
        """
        Return the top_k completions for prefix, ranked by frequency.
        Returns a list of (word, count) tuples.
        """
        candidates = self.trie.autocomplete(prefix)
        # Use heapq.nlargest for efficient top-k selection
        return heapq.nlargest(
            top_k,
            ((w, self.counter[w]) for w in candidates),
            key=lambda x: x[1]
        )


# Build from a simulated search log
ac = AutoComplete()
search_log = [
    ('python',        150), ('python tutorial', 80), ('python install',   45),
    ('python list',    60), ('python dict',      55), ('python class',     40),
    ('pytorch',        90), ('pypy',              20), ('pytest',           70),
    ('pycharm',        35), ('pizza recipe',      95), ('pizza delivery',  110),
    ('pizza near me', 200), ('pi value',          15), ('pie chart',        30),
]
for word, count in search_log:
    ac.add(word, count)

print("Suggestions for 'py':")
for word, count in ac.suggest('py', top_k=5):
    print(f"  {word:<25} ({count:,} searches)")

print("\nSuggestions for 'pi':")
for word, count in ac.suggest('pi', top_k=3):
    print(f"  {word:<25} ({count:,} searches)")